# Note: 
1. Remember, tools binding is available in Chat models only, so `OpenAI` and `GoogleGenerativeAI` wil not work.
2. Not all models have capabilities to bind tools. We will see which all models can.

In [2]:
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

from dotenv import load_dotenv

In [3]:
load_dotenv()

True

# Simple example to bind a tool with an LLM

In [4]:
from typing import Union

In [19]:
@tool
def multiply(a:Union[int, float], b:Union[int, float], c:Union[int, float] = 1 ) ->Union [int, float]:
    '''Given two numbers a and b, this tool will return theirs product.
    Third number c is optional. Cannot accept more than three numbers'''
    return a*b*c

# Instead of Union[int, float] we can also use shortcut int| float

In [20]:
print(multiply.invoke({'a': 5, 'b': 2}))
print()

print(multiply.args)
print()

print(multiply.description)
print()

print(multiply.name)
print()

10

{'a': {'anyOf': [{'type': 'integer'}, {'type': 'number'}], 'title': 'A'}, 'b': {'anyOf': [{'type': 'integer'}, {'type': 'number'}], 'title': 'B'}, 'c': {'anyOf': [{'type': 'integer'}, {'type': 'number'}], 'default': 1, 'title': 'C'}}

Given two numbers a and b, this tool will return theirs product.
    Third number c is optional. Cannot accept more than three numbers

multiply



In [21]:
# Creating an LLM

model_gemini = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash'
)

#Binding the tool
llm_with_tools = model_gemini.bind_tools([multiply])

print(llm_with_tools)

bound=ChatGoogleGenerativeAI(model='models/gemini-2.5-flash', google_api_key=SecretStr('**********'), client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x0000024F3AECAC50>, default_metadata=()) kwargs={'tools': [{'type': 'function', 'function': {'name': 'multiply', 'description': 'Given two numbers a and b, this tool will return theirs product.\n    Third number c is optional. Cannot accept more than three numbers', 'parameters': {'properties': {'a': {'anyOf': [{'type': 'integer'}, {'type': 'number'}]}, 'b': {'anyOf': [{'type': 'integer'}, {'type': 'number'}]}, 'c': {'anyOf': [{'type': 'integer'}, {'type': 'number'}], 'default': 1}}, 'required': ['a', 'b'], 'type': 'object'}}}]} config={} config_factories=[]


# Tool Calling:

1. `Tool Calling` is the process where the LLM decides during an conversation, that it need to use a specific tool
2. LLM generates a structured output with ` the name of the tool` and `the arguments to call it`, when it calls any tool

For example: If we type, `What is 8*5?` <br>
LLM will first search for tool which can multipll the given numbers, then it will fo through its description and s=generate output in below format. <br>

`{'name': 'multiply',`<br>
`'args': 'a': 8, 'b': 5}`

# Let's continue with our created model named `llm_with_tools` and see how it calls the tool

In [22]:
response = llm_with_tools.invoke("How are you?")

print(response)

# See, there is no schema for tool calling

content="I'm doing well, thank you for asking! I'm ready to assist you with any questions or tasks you have. How can I help you today?" additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []} id='run--17b5b2dd-cced-435f-8da4-b0cf991cc69e-0' usage_metadata={'input_tokens': 89, 'output_tokens': 33, 'total_tokens': 122, 'input_token_details': {'cache_read': 0}}


In [ ]:
response = llm_with_tools.invoke("What is 8*5?")
print(response)

# See, here there is an attribute 'tool_calling' where schema is present and content is blank
# There can be many tool_calls as per the requirement

content='' additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 8.0, "b": 5.0}'}} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []} id='run--8045009e-28ff-46df-91d4-60bbd32826b4-0' tool_calls=[{'name': 'multiply', 'args': {'a': 8.0, 'b': 5.0}, 'id': '6fda659c-1888-4c58-95e0-322bfa0bcbcf', 'type': 'tool_call'}] usage_metadata={'input_tokens': 92, 'output_tokens': 18, 'total_tokens': 197, 'input_token_details': {'cache_read': 0}}


In [24]:
response.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 8.0, 'b': 5.0},
 'id': '6fda659c-1888-4c58-95e0-322bfa0bcbcf',
 'type': 'tool_call'}

# Note:
See, we thought than an LLM will call the tool and return the result for `8 * 5 = 40`. But LLM doesn't actually calls the tool, it just tells us that a particular tool should be called with its `name` and its `input parameters` <br>

The actual tool execution is done by the programmer or the langchain.

# Tool Execution:
Tool execution is the process where the actual python function (tool here) is executed using the input arguments that the LLM suggested on tool calling.

In simple words <br>

`LLM says:` <br>
Hey, call the multiply tool with a = 8 and b = 5

Tool execution is when the programmer/ Langchain actally runs <br>
`multiply({'a': 8, 'b': 5})`

`Let's continue from the tool call LLM suggested`

In [25]:
response.tool_calls[0]

# See, args is the exact input we have to pass to our function.
# Very simple, just pass it to multiply()

{'name': 'multiply',
 'args': {'a': 8.0, 'b': 5.0},
 'id': '6fda659c-1888-4c58-95e0-322bfa0bcbcf',
 'type': 'tool_call'}

In [33]:
print(multiply.invoke(response.tool_calls[0]['args']))

40.0


In [34]:
multiply.invoke(response.tool_calls[0])

ToolMessage(content='40.0', name='multiply', tool_call_id='6fda659c-1888-4c58-95e0-322bfa0bcbcf')

Obviously, if we pass only args, it will simply return the answer but <br>
if we pass the entire dict schema of tool_call, it returns us an object of <br>
ToolMessage. This is same like HumanMessage, SystemMessage and AIMessage <br>

Now, what we can do is, we can pass this ToolMessage object to the LLM with a prompt saying, <br>
"We got this result upon execution on the tool with the inputs you suggested."

For this, we will have to maintain conversation's history in the form of HumanMessage, AIMessage, ToolMessage objects

In [46]:
from langchain_core.messages import HumanMessage, AIMessage

In [63]:
# Creating a tool
@tool
def multiply(a:int|float, b:int|float, c:int|float = 1) ->int|float:
    '''This function takes three integer or float arguments(2 compulsory)
    and returns theirs product'''
    return a*b*c

# Initializing the model
model_gemini = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash'
)

# Binding tool with the model
model_with_tool = model_gemini.bind_tools(tools = [multiply])

# Maintining the history
hist = [HumanMessage(content = "What is 8*5?")]

# Passing our query to the LLM
response = model_with_tool.invoke(hist)

# Adding model's response to the history.
# Model give its response in the form of the object of AIMessage only

hist.append(response)

# Extraction tool_calls parth from the response
tool_call = response.tool_calls[0]

# Tool execution
tool_message = multiply.invoke(tool_call)

# adding tool_message into the history
hist.append(tool_message)

# passing history to the LLM

response2 = model_with_tool.invoke(hist)

In [64]:
print(response2)
print()

print(f'Content: {response2.content}')

content='The product of 8 and 5 is 40.' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []} id='run--a760a742-54e6-43ba-9d24-cba44e460633-0' usage_metadata={'input_tokens': 114, 'output_tokens': 13, 'total_tokens': 127, 'input_token_details': {'cache_read': 0}}

Content: The product of 8 and 5 is 40.


# Thats it, such simple the tool calling is.

# Next session: Real time currency conversion tool